# Experiment 1 — <Strategy_Name> (the benchmark)

**The pure idea, and the yardstick every later experiment is measured against.**

> **<The main idea from `OBJECTIVE.md`, in one sentence.>**

> **Run order: step 6 of 6** (see [`../../README.md`](../../README.md)). Last. It reads the refined
> panel and writes a book; sections 4 and 5 of this notebook need the licensed engines and
> report-and-skip without them.

The rule in section 2 is the simplest thing that uses a signal at all: hold everything the signal
calls eligible, in equal weight, and hold cash for the rest. **It runs as soon as you name your
eligibility column in `SIGNAL_COLUMN`** — deliberately, because a benchmark you have to write before
you can measure anything is a benchmark that never gets written.

| Decision | Rule |
| --- | --- |
| **Selection** | `<SIGNAL_COLUMN> == 1.0` |
| **Ranking** | none. Without a holding cap the eligible set *is* the book |
| **Weighting** | equal, across whatever is eligible. No cap, no minimum |
| **Rebalancing** | event-driven: only on days the eligible set changes |
| **Lag** | decide on yesterday's close, fill at today's VWAP |
| **Residual** | parked in the cash proxy, the tradable stand-in for cash |

The hypothesis and the predictions this is testing are in
[`BLUEPRINT_1.md`](BLUEPRINT_1.md) — **written before this notebook's rule cell.** The running log is
[`JOURNAL_1.md`](JOURNAL_1.md), and the results that survive are in [`FINDINGS_1.md`](FINDINGS_1.md).

## Position in the pipeline

This notebook is **only the strategy**. The universe and the data are built by earlier stages and
are simply read here:

```
Universe/universe.ipynb   ->  Security_Master.csv
Data/curator.py           ->  Data/Curator/Time_Series/     m_* + c_*
Data/refinery.py          ->  Data/Refinery/Time_Series/    + r_*      <- this notebook reads here
Data/analyzer.ipynb       ->  the measurements the blueprint's predictions came from
        |
        v
experiment_1.ipynb        ->  Portfolio/  ->  Backtest/  ->  Attribution/
```

## What this notebook does *not* do

It does not download anything, profile the universe, or compute a signal. If a number about the
data itself is needed, it belongs in the Universe or Data stage — that separation is what keeps
every experiment comparable, because all of them read the identical panel.

**The template's contract.** Every section below is strategy-agnostic except **section 2, the
rule**, which is the one cell you write. It must produce three objects — `selected_matrix`,
`REBALANCE_DATES`, `target_weights` — and everything downstream runs unchanged.

---

## 0 · Setup

Paths, the strategy's columns, and the panel. Only the columns the strategy consumes are read.

The three price columns do **three different jobs**, and getting them out of step is silent — the
backtest P&L and the attribution would quietly run on different bases:

| Role | Column | Why |
| --- | --- | --- |
| Daily mark | `m_close_dividend_and_split_adjusted` | total-return valuation between rebalances |
| Fill | `c_vwap_dividend_and_split_adjusted` | the price a trade actually gets |
| Commission | `c_vwap` | per-share cents ride on the **unadjusted** share count |

The provider returns `m_vwap` and `m_vwap_dividend_and_split_adjusted` as null, which is exactly why
the Curator reconstructs them as `c_*`. Never point anything at the `m_vwap*` columns.

**This is the benchmark, so the loading steps are written inline** rather than imported from
`Experiments/securities_panel.py`: a baseline that cannot be read top to bottom without chasing an import is a
worse baseline. Later experiments import `securities_panel.py` so their notebooks show only what they change.

In [ ]:
"""Experiment 1 - the benchmark. Reads Data/Refinery/, writes this experiment's folders."""
import pathlib
import sys

import matplotlib.pyplot
import numpy
import pandas


def find_repo_root(start):
    """Walk up from `start` to the directory that holds pyproject.toml and Experiments/."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "Experiments").is_dir():
            return candidate

    return start


REPO_ROOT = find_repo_root(pathlib.Path.cwd())
sys.path.insert(0, str(REPO_ROOT / "Experiments"))
import attribution_analysis
import backtest_engine
import portfolio_construction

EXPERIMENT_DIR = REPO_ROOT / "Experiments" / "Experiment_1"
REFINERY_DIR = REPO_ROOT / "Data" / "Refinery" / "Time_Series"
# The engine reads one market-data directory and needs the cash proxy and the benchmarks in it,
# and those are deliberately absent from the Refinery (they are not part of any cross-section).
MARKET_DATA_DIR = REPO_ROOT / "Data" / "Curator" / "Time_Series"
BENCHMARK_DIR = REPO_ROOT / "Data" / "Curator" / "Benchmarks"
UNIVERSE_PATH = REPO_ROOT / "Universe" / "Investable_Universe.csv"

PORTFOLIO_DIR = EXPERIMENT_DIR / "Portfolio"
BACKTEST_DIR = EXPERIMENT_DIR / "Backtest"
ATTRIBUTION_DIR = EXPERIMENT_DIR / "Attribution"
CHART_DIR = PORTFOLIO_DIR / "Charts"
for directory in (PORTFOLIO_DIR, BACKTEST_DIR, ATTRIBUTION_DIR, CHART_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# --- The strategy's columns ----------------------------------------------------------------
# The only constants in this notebook that belong to the strategy rather than to the process.
# They are declared here, not in Experiments/securities_panel.py, so a signal never becomes every
# later experiment's default without anyone deciding it.
DATE_COLUMN = "m_date"
MARK_PRICE_COLUMN = "m_close_dividend_and_split_adjusted"
TRADE_PRICE_COLUMN = "c_vwap_dividend_and_split_adjusted"
CLASSIFICATION_COLUMN = "asset_group_current"
# `SIGNAL_COLUMN` is the 0/1 eligibility column your strategy selects on. `REPORTING_COLUMN` is
# carried into the deliverables so a person reading the book can see how each holding ranked against
# the others - **the rule does not read it**, because Experiment 1 weights equally and so ranks
# nothing. It is here to make the next experiment's question visible, not to answer it.
SIGNAL_COLUMN = "<r_your_eligibility_column>"
REPORTING_COLUMN = "r_liquidity_rank"

assert not SIGNAL_COLUMN.startswith("<"), (
    "name your eligibility column in SIGNAL_COLUMN above - a 0/1 column from the Refinery saying"
    " whether a security may be held. Everything below runs once it is set."
)

PANEL_COLUMNS = (
    DATE_COLUMN,
    SIGNAL_COLUMN,
    REPORTING_COLUMN,
    MARK_PRICE_COLUMN,
    TRADE_PRICE_COLUMN,
    CLASSIFICATION_COLUMN,
)
CASH_TICKER = backtest_engine.CASH_TICKER

paths = sorted(REFINERY_DIR.glob("*.csv"))
assert paths, f"no refined files in {REFINERY_DIR} - run: uv run python Data/refinery.py"

frames = []
for path in paths:
    frame = pandas.read_csv(path, usecols=list(PANEL_COLUMNS), parse_dates=[DATE_COLUMN])
    frame.insert(0, "ticker", path.stem)
    frames.append(frame)

panel = pandas.concat(frames, ignore_index=True)

print(f"Repo root : {REPO_ROOT}")
print(f"Reading   : {REFINERY_DIR.relative_to(REPO_ROOT)}")
print(f"Panel     : {panel['ticker'].nunique()} securities x {panel[DATE_COLUMN].nunique()} dates"
      f" = {len(panel):,} rows")
print(f"Window    : {panel[DATE_COLUMN].min().date()} -> {panel[DATE_COLUMN].max().date()}")
print(f"Signal    : {SIGNAL_COLUMN}, present on {panel[SIGNAL_COLUMN].notna().mean():.0%} of rows")

---

## 1 · One position per company, then reshape to matrices

The universe is point-in-time, so it contains **ticker changes**: two legs sharing an ISIN, each
carrying only part of the history. Left alone they would be two independent positions and the book
would double-count the company at the changeover.

Positions are therefore keyed by **ISIN**, falling back to the ticker where the universe file has
none. Where two legs overlap on a date, the leg still reporting later wins — that is the surviving
listing.

The long panel then becomes one wide `dates × companies` matrix per input, which is what makes the
whole selection rule in section 2 a handful of vectorised lines instead of a loop over files.

In [ ]:
universe = pandas.read_csv(UNIVERSE_PATH, encoding="utf-8-sig", dtype=str)
universe["ticker"] = universe["ticker"].str.strip()
# Only `ticker` and `name` are required of a seed. Anything else this notebook would like is
# filled in as null rather than assumed, so a minimal universe file still loads.
for optional in ("isin", "asset_class"):
    if optional not in universe.columns:
        universe[optional] = None

# Company key: ISIN where the universe file has one, the ticker itself otherwise.
company_by_ticker = {
    row.ticker: (row.isin if isinstance(row.isin, str) and row.isin else row.ticker)
    for row in universe.itertuples()
}
panel["company"] = panel["ticker"].map(company_by_ticker).fillna(panel["ticker"])

tickers_by_company = {}
for ticker, company in company_by_ticker.items():
    tickers_by_company.setdefault(company, []).append(ticker)
multi_leg_companies = {
    company
    for company, ticker_list in tickers_by_company.items()
    if len(ticker_list) > 1
}

# Resolve overlaps: sort so the leg that reports latest sorts last, then keep the last row.
last_date_by_ticker = panel.groupby("ticker")[DATE_COLUMN].max()
panel["leg_rank"] = panel["ticker"].map(last_date_by_ticker)
overlapping_rows = int(
    panel[panel["company"].isin(multi_leg_companies)]
    .duplicated(subset=["company", DATE_COLUMN], keep=False)
    .sum()
)

stitched = (
    panel.sort_values(["company", DATE_COLUMN, "leg_rank"])
    .drop_duplicates(subset=["company", DATE_COLUMN], keep="last")
    .drop(columns="leg_rank")
)

company_metadata = (
    stitched.sort_values(DATE_COLUMN)
    .groupby("company")
    .agg(
        ticker=("ticker", "last"),
        asset_group=(CLASSIFICATION_COLUMN, "last"),
    )
)
name_by_isin = dict(zip(universe["isin"], universe["name"]))
class_by_isin = dict(zip(universe["isin"], universe["asset_class"]))
company_metadata["name"] = pandas.Series(
    company_metadata.index.map(name_by_isin),
    index=company_metadata.index,
).fillna(company_metadata["ticker"])
company_metadata["asset_class"] = pandas.Series(
    company_metadata.index.map(class_by_isin),
    index=company_metadata.index,
).fillna(company_metadata["ticker"])


def to_matrix(frame, column):
    """A dates x companies matrix of `column`, over the union of all observed trading days."""

    return frame.pivot(index=DATE_COLUMN, columns="company", values=column).sort_index()


signal_matrix = to_matrix(stitched, SIGNAL_COLUMN)
reporting_matrix = to_matrix(stitched, REPORTING_COLUMN)
mark_price_matrix = to_matrix(stitched, MARK_PRICE_COLUMN)
trade_price_matrix = to_matrix(stitched, TRADE_PRICE_COLUMN)
ticker_matrix = to_matrix(stitched, "ticker")

# Daily total returns, which is what any risk-aware weighting scheme estimates from. Experiment 1
# does not use them; building them here keeps the rule cell identical when a later experiment does.
return_matrix = mark_price_matrix.pct_change()

TRADING_DAYS = signal_matrix.index

print(f"stitched : {len(panel):,} ticker-rows -> {len(stitched):,} company-rows")
print(f"           {stitched['company'].nunique()} securities"
      f" from {panel['ticker'].nunique()} tickers"
      f" ({len(multi_leg_companies)} multi-leg, {overlapping_rows:,} overlapping rows resolved)")
print(f"matrices : {signal_matrix.shape[0]:,} trading days x {signal_matrix.shape[1]} securities")
print()
print(company_metadata[["ticker", "asset_class", "asset_group"]].to_string(index=False))

---

## 2 · The rule — the one cell you write

Three statements, in order, each a whole-matrix operation so the strategy is defined without a loop
over dates:

1. **Who is eligible.** The signal is on *and* the asset is tradable — a mark and a fill price both
   exist today.
2. **How much of each.** Handed to a **weigher** in `Experiments/portfolio_construction.py`, which
   is the seam the KaxaNuk Portfolio Construction library will replace. Swapping
   `portfolio_construction.equal_weight` for `inverse_volatility`, or for a call into that library,
   is a one-line change and nothing else in this notebook moves.
3. **When to trade.** Only on days the eligible set changes. A signal that has not moved is not a
   reason to pay commission.

**The contract this cell must satisfy** — everything below reads exactly these three objects:

| Object | Type | Meaning |
| --- | --- | --- |
| `selected_matrix` | `dates × securities` boolean | what the book holds on each day |
| `REBALANCE_DATES` | `DatetimeIndex` | the days the book is re-struck |
| `target_weights` | `REBALANCE_DATES × securities` float, rows summing to **at most** 1.0 | the book on each of those days |

**Rows sum to at most one, not to exactly one.** A book that must be fully invested cannot express
a defensive strategy, and this one is defensive by construction: when nothing is eligible it holds
nothing. The residual becomes cash in section 3.1, parked in a real priced instrument, because the
engine's weight file has no cash row of its own.

### Two look-aheads, both stated plainly

**The lag.** The eligible set used on rebalance date *t* is the one observed at *t−1*, and the
engine fills at *t*'s VWAP — a full day between the signal and the fill.
`portfolio_construction.lag_eligibility` is that rule in one call, so it cannot be forgotten.

**The delisting exit.** A security that delists must be sold on the **last day it still has a fill
price**, and knowing that day is its last requires seeing the next one. This is the standard
backtest compromise — the alternative, carrying a position that can never be exited, is a larger
distortion — and it is implemented by making a name ineligible on that final day, so the set
changes, the rebalance fires, and the position is sold while a price still exists. Inert on this
universe, which is twelve live ETFs; **load-bearing on any universe that retains delisted names.**

In [ ]:
# --- Process: tradability and the documented look-ahead (do not change) ----------------------
tradable_matrix = mark_price_matrix.notna() & trade_price_matrix.notna()

# The last day a security can still be sold. `shift(-1)` is the documented look-ahead; the final
# row of the sample is excluded because the end of the data is not a delisting.
last_tradable_day = tradable_matrix & ~tradable_matrix.shift(-1, fill_value=False)
last_tradable_day.iloc[-1] = False

# --- Strategy: the rule ----------------------------------------------------------------------
SIGNAL_LAG_DAYS = 1                                   # decide on yesterday's close
WEIGHER = portfolio_construction.equal_weight         # <- swap this line to change the sizing
SETTINGS = portfolio_construction.ConstructionSettings()   # every constraint off, on purpose

# 1. Eligible: the signal was on at the prior close, and the asset can be traded today.
signal_active = signal_matrix == 1.0
buyable_matrix = (
    portfolio_construction.lag_eligibility(signal_active, SIGNAL_LAG_DAYS)
    & tradable_matrix
    & ~last_tradable_day
)

# 2. When: only when the set changes.
REBALANCE_DATES = portfolio_construction.rebalance_dates_on_change(buyable_matrix)

# 3. How much: handed to the weigher, which never sees a date and so cannot reach into the future.
target_weights = portfolio_construction.build_target_weights(
    buyable_matrix,
    REBALANCE_DATES,
    return_matrix,
    WEIGHER,
    SETTINGS,
)

# What the book holds between rebalances: the last struck book, carried forward. Carried as
# floats and compared afterwards, because forward-filling a boolean frame goes through object
# dtype and pandas is deprecating the silent downcast back.
selected_matrix = target_weights.reindex(TRADING_DAYS).ffill().fillna(0.0) > 0.0

invested = target_weights.sum(axis=1)
print(f"Weigher       : {WEIGHER.__name__}  {SETTINGS}")
print(f"Rebalances    : {len(REBALANCE_DATES):,}"
      f" from {REBALANCE_DATES[0].date()} to {REBALANCE_DATES[-1].date()}")
print(f"Holdings      : {(target_weights > 0).sum(axis=1).mean():.1f} on average,"
      f" {(target_weights > 0).sum(axis=1).max()} at most")
print(f"Invested      : {invested.mean():.1%} on average,"
      f" {invested.min():.0%} at the least, {invested.max():.0%} at the most")
print(f"Fully in cash : {(invested < 1e-9).sum()} of {len(REBALANCE_DATES)} rebalances")

### 2.1 · Invariants

Cheap to check here, expensive to discover inside a P&L. **Every rule must pass these unchanged.**

In [ ]:
TOLERANCE = 1e-9

row_sums = target_weights.sum(axis=1)
assert (row_sums <= 1.0 + TOLERANCE).all(), (
    f"a book cannot be more than fully invested; worst = {row_sums.max():.9f}"
)
assert (row_sums >= -TOLERANCE).all(), "a book cannot be negatively invested"
assert (target_weights >= -TOLERANCE).all().all(), "no negative weights: the strategy is long-only"

held = target_weights > 0
assert (held.sum(axis=1) <= signal_matrix.shape[1]).all(), "more securities held than exist"

# No look-ahead: every security paid for today had its signal on at *yesterday's* close. This
# checks the signal itself rather than the composed eligibility, because the signal is the thing
# that had to exist in advance - tradability is a property of the day the trade happens.
signal_yesterday = signal_active.shift(1, fill_value=False).loc[REBALANCE_DATES]
assert not (held & ~signal_yesterday).any().any(), (
    "a weight was assigned to a security whose signal was not on at the prior close"
)

# Every security bought is tradable on the day it is bought, so the fill price exists.
assert not (held & ~tradable_matrix.loc[REBALANCE_DATES]).any().any(), (
    "a weight was assigned to a security with no fill price on its implementation date"
)

# Nothing is still held on a day after it stopped being tradable.
carried = held.astype("boolean").reindex(TRADING_DAYS).ffill().fillna(False).astype(bool)
stranded = carried & ~tradable_matrix
stranded_names = stranded.any(axis=0)
assert not stranded_names.any(), (
    f"{int(stranded_names.sum())} position(s) carried past their last tradable day"
)

print("All invariants hold:")
print(f"  every book is between 0% and 100% invested (mean {row_sums.mean():.1%})")
print("  no negative weights")
print("  every holding had its signal on at the prior close and a fill price when bought")
print("  no position is carried past its last tradable day")

---

## 3 · Construction — is this a book you would actually run?

Four properties, each with a failure mode a performance chart would hide.

| Property | What a bad value would mean |
| --- | --- |
| Invested share over time | the eligibility column is not doing what the analyzer says it does |
| Trigger frequency and turnover | the rule fires so often that this is a transaction-cost question, not an alpha one |
| Holdings and concentration | a "diversified" label on a book that is one or two positions |
| Asset-group drift | the strategy is a disguised bet on one group rather than a rotation between them |

Turnover is measured **target-to-target**. The realised figure is slightly lower, because between
rebalances the winners drift up on their own; that calculation needs drifted weights and belongs to
the backtest.

> **The predictions in [`BLUEPRINT_1.md`](BLUEPRINT_1.md) that are about the *shape* of the book,
> rather than about its return, are settled here — before any backtest.** They are the first ones
> that can be wrong, and the cheapest to be wrong about.

**This is where step 4, Portfolio Construction, lives.** It is `portfolio_construction.py` today —
`equal_weight` and `inverse_volatility` behind one signature. When the KaxaNuk Portfolio
Construction library lands, a wrapper of that shape goes in beside them and `WEIGHER` in section 2
points at it. Nothing else in this notebook changes.

In [ ]:
turnover = (target_weights.diff().abs().sum(axis=1) / 2.0).rename("turnover_one_way")
turnover.iloc[0] = numpy.nan  # The first book is an initial build, not a rebalance.

invested_share = target_weights.sum(axis=1).rename("invested_share")
effective_names = (
    (1.0 / (target_weights ** 2).sum(axis=1)).replace(numpy.inf, 0.0).rename("effective_names")
)

portfolio_summary = pandas.concat(
    [
        (target_weights > 0).sum(axis=1).rename("holdings"),
        invested_share,
        turnover,
        effective_names,
        target_weights.max(axis=1).rename("max_weight"),
    ],
    axis=1,
)
portfolio_summary.index.name = "rebalance_date"

years = (REBALANCE_DATES[-1] - REBALANCE_DATES[0]).days / 365.25
print("Per-rebalance summary:")
print(portfolio_summary.describe().loc[["mean", "min", "max"]].round(3).to_string())
print(f"\n{len(REBALANCE_DATES):,} rebalances over {years:.1f} years"
      f" = {len(REBALANCE_DATES) / years:.0f} a year;"
      f" one-way turnover ~{turnover.sum() / years:.0%} a year")

# --- How much of the book is cash, and how much does it move? ---------------------------------
# Weighted by calendar days, not by rebalance count, because a book held for six months matters
# more than one held for a week.
daily_invested = invested_share.reindex(TRADING_DAYS).ffill().dropna()
fully_invested = daily_invested >= 0.999
longest_stretch = int(
    fully_invested.astype(int).groupby((~fully_invested).cumsum()).sum().max()
)
print("\nHow invested is it, and how much does that move?")
print(f"  invested, averaged over trading days : {daily_invested.mean():.1%}")
print(f"  days at least 90% invested           : {(daily_invested >= 0.9).mean():.1%}")
print(f"  days at most 25% invested            : {(daily_invested <= 0.25).mean():.1%}")
print(f"  longest fully-invested stretch       : {longest_stretch} trading days")

# --- Asset-group exposure against the universe's own composition -------------------------------
group_by_company = company_metadata["asset_group"].fillna("(unclassified)")
group_weights = (
    target_weights.T.groupby(group_by_company.reindex(target_weights.columns).to_numpy())
    .sum().T
)
group_weights.index.name = "rebalance_date"

universe_group_share = (
    group_by_company.reindex(target_weights.columns).value_counts(normalize=True)
    .reindex(group_weights.columns).fillna(0.0)
)
average_group_weight = group_weights.mean()
group_drift = pandas.DataFrame({
    "portfolio": average_group_weight,
    "universe_equal_weight": universe_group_share,
    "drift_pp": (average_group_weight - universe_group_share) * 100,
}).sort_values("drift_pp", ascending=False)

print("\nAverage weight by asset group, against an always-invested equal-weight book:")
print(group_drift.round(3).to_string())
print("\n-> the portfolio column sums to the average invested share, not to 1.0;"
      " the gap is cash.")

In [ ]:
INK_SECONDARY = "#52514e"
SERIES_BLUE = "#2a78d6"
SERIES_ORANGE = "#eb6834"
SURFACE = "#fcfcfb"
GRID = "#ebeae5"

matplotlib.pyplot.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "axes.edgecolor": "#d8d7d2",
    "axes.labelcolor": INK_SECONDARY,
    "xtick.color": INK_SECONDARY,
    "ytick.color": INK_SECONDARY,
    "font.size": 10,
    "figure.dpi": 110,
    "savefig.dpi": 160,
    "savefig.bbox": "tight",
})


def style_axes(axes, title=None, subtitle=None, ylabel=None):
    """Left-aligned bold title over an optional subtitle line, with a recessive grid."""
    if title:
        axes.set_title(title, loc="left", pad=22 if subtitle else 8, weight="bold")
    if subtitle:
        axes.annotate(
            subtitle, xy=(0, 1), xycoords="axes fraction",
            xytext=(0, 6), textcoords="offset points",
            fontsize=9, color=INK_SECONDARY, va="bottom", ha="left",
        )
    if ylabel:
        axes.set_ylabel(ylabel)
    axes.grid(True, color=GRID, linewidth=0.8)
    axes.set_axisbelow(True)
    for side in ("top", "right"):
        axes.spines[side].set_visible(False)

    return axes


# --- How much of the book is invested, held flat between rebalances --------------------------
figure, axes = matplotlib.pyplot.subplots(figsize=(13, 4.2))
axes.fill_between(daily_invested.index, daily_invested.to_numpy(), color=SERIES_BLUE, alpha=0.25)
axes.plot(daily_invested.index, daily_invested.to_numpy(), color=SERIES_BLUE, linewidth=1.1)
axes.axhline(daily_invested.mean(), color=SERIES_ORANGE, linewidth=1.4, linestyle="--")
axes.set_ylim(0, 1)
axes.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
style_axes(
    axes, "How much of the book is invested",
    subtitle=f"the rest is cash; the dashed line is the average, {daily_invested.mean():.0%}",
    ylabel="invested share",
)
figure.tight_layout()
figure.savefig(CHART_DIR / "portfolio_invested_share.png")
matplotlib.pyplot.show()

# --- The book over time ------------------------------------------------------------------------
panels = (
    (portfolio_summary["holdings"], "Holdings", "securities", False),
    (portfolio_summary["turnover_one_way"], "One-way turnover per rebalance", "share", True),
    (portfolio_summary["max_weight"], "Largest single weight", "share of book", True),
    (portfolio_summary["effective_names"], "Effective number of names (1 / HHI)", "names", False),
)

figure, axes_grid = matplotlib.pyplot.subplots(2, 2, figsize=(13, 6.4), sharex=True)
for axes, (series, title, ylabel, is_share) in zip(axes_grid.flatten(), panels):
    axes.plot(series.index, series.to_numpy(), color=SERIES_BLUE, linewidth=1.1)
    if is_share:
        axes.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
    style_axes(axes, title, ylabel=ylabel)
figure.suptitle(
    "Experiment 1 book - equal weight over the eligible set",
    x=0.01, ha="left", fontsize=13, weight="bold",
)
figure.tight_layout(rect=(0, 0, 1, 0.95))
figure.savefig(CHART_DIR / "portfolio_book_over_time.png")
matplotlib.pyplot.show()

# --- Where the book actually sits, group by group ---------------------------------------------
daily_groups = group_weights.reindex(TRADING_DAYS).ffill().dropna(how="all")
figure, axes = matplotlib.pyplot.subplots(figsize=(13, 4.6))
axes.stackplot(
    daily_groups.index,
    *[daily_groups[column].to_numpy() for column in daily_groups.columns],
    labels=list(daily_groups.columns),
    alpha=0.85,
)
axes.set_ylim(0, 1)
axes.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
axes.legend(frameon=False, fontsize=9, ncol=3, loc="upper left")
style_axes(
    axes, "The book by asset group",
    subtitle="whatever is missing from the top of the stack is cash",
    ylabel="weight",
)
figure.tight_layout()
figure.savefig(CHART_DIR / "portfolio_group_weights.png")
matplotlib.pyplot.show()

# --- The latest book ---------------------------------------------------------------------------
latest = target_weights.iloc[-1]
latest = latest[latest > 0].sort_values(ascending=False)
print(f"The book on {target_weights.index[-1].date()}"
      f" - {len(latest)} holdings, {latest.sum():.0%} invested:")
if len(latest) > 0:
    print(
        pandas.DataFrame({
            "ticker": ticker_matrix.loc[target_weights.index[-1], latest.index],
            "asset_class": company_metadata.loc[latest.index, "asset_class"],
            "weight": latest.map("{:.2%}".format),
        }).to_string(index=False)
    )
print(f"cash ({CASH_TICKER}): {1.0 - latest.sum():.2%}")

### 3.1 · Write the deliverables

Two views of the same book, because two different readers need it:

- **`target_weights.csv`** — long, company-keyed, with names and asset classes attached. The
  research view, and what a human reads.
- **`portfolio_weights.csv`** — wide, **ticker**-keyed, dates across the columns. The backtest
  engine's input: it looks up `<Ticker>.csv` in the market-data directory, so it has to speak in
  tickers, not in the ISIN-stitched companies. A security that changed ticker mid-history occupies
  two rows, each non-zero only while that listing was live.

**This is where cash becomes a position.** Everything above lets a book be less than fully
invested; `to_engine_frame` turns that residual into a weight in `BIL`, so the engine charges
commission on going to cash and earns the bill yield while there. A strategy whose defining move is
*sell everything* has to pay for it.

In [ ]:
held_weights = target_weights.stack()
held_weights = held_weights[held_weights > 0]

target_weights_long = (
    pandas.concat(
        {
            "weight": held_weights,
            "ticker": ticker_matrix.loc[REBALANCE_DATES].stack().reindex(held_weights.index),
            REPORTING_COLUMN: reporting_matrix.shift(1).loc[REBALANCE_DATES].stack()
                                              .reindex(held_weights.index),
        },
        axis=1,
    )
    .rename_axis(["rebalance_date", "company"])
    .reset_index()
    .join(company_metadata[["name", "asset_class", "asset_group"]], on="company")
    .sort_values(["rebalance_date", "weight"], ascending=[True, False])
)

# Shaped by the shared helper, so this file and the one section 4 hands the engine are built by
# one code path. The engine's weight file has no cash row -- every column must sum to 1.0 -- so
# the residual is parked in the cash proxy, keeping "go to cash" expressible rather than
# structurally impossible.
engine_weights = backtest_engine.to_engine_frame(target_weights, ticker_matrix, CASH_TICKER)
cash_weight = (
    engine_weights.loc[CASH_TICKER]
    if CASH_TICKER in engine_weights.index
    else pandas.Series(0.0, index=engine_weights.columns)
)

column_sums = engine_weights.sum(axis=0)
assert numpy.allclose(column_sums, 1.0, atol=1e-8), (
    f"engine weight columns must sum to 1.0; worst = {column_sums.min():.9f}"
)

missing_price_files = [
    ticker for ticker in engine_weights.index
    if not (MARKET_DATA_DIR / f"{ticker}.csv").is_file()
]
assert not missing_price_files, (
    f"no price file in {MARKET_DATA_DIR} for: {', '.join(missing_price_files[:10])}"
)

target_weights_long.to_csv(PORTFOLIO_DIR / "target_weights.csv", index=False)
portfolio_summary.to_csv(PORTFOLIO_DIR / "portfolio_summary.csv")
group_weights.to_csv(PORTFOLIO_DIR / "group_weights.csv")
engine_weights.to_csv(PORTFOLIO_DIR / "portfolio_weights.csv")

print(f"Written to {PORTFOLIO_DIR.relative_to(REPO_ROOT)}:")
print(f"  portfolio_weights.csv  {engine_weights.shape[0]:,} tickers x"
      f" {engine_weights.shape[1]:,} dates   <- backtest engine input")
print(f"  target_weights.csv     {len(target_weights_long):,} rows (rebalance x holding)")
print(f"  portfolio_summary.csv  {len(portfolio_summary):,} rows")
print(f"  group_weights.csv      {group_weights.shape[0]:,} x {group_weights.shape[1]}")
print("  Charts/                4 PNGs")
print(f"\nCash weight ({CASH_TICKER}): mean {cash_weight.mean():.1%},"
      f" min {cash_weight.min():.1%}, max {cash_weight.max():.1%}")

---

## 4 · Backtest — KaxaNuk Backtest Engine

`portfolio_weights.csv` goes to the licensed engine, which simulates the book share by share: it
fills at a real price, charges commission per share on the unadjusted price, holds a cash reserve,
marks the portfolio daily between rebalances, and compares against the benchmarks.

**This is the only backtest in the repository.** There is deliberately no second, lighter
simulator: a simpler backtest that disagrees with the engine is worse than none at all, because it
lets the reader pick whichever number they prefer. Every figure quoted anywhere — here, in later
experiments, in `RESULTS.md` — comes from this engine.

Benchmarks are **SPY**, **QQQ** and **KN600** (the KaxaNuk US equity index). The window is clipped
to the shortest benchmark up front rather than discovered as a crash: the engine validates that
every benchmark spans the whole period and raises otherwise.

> **Environment note.** `kaxanuk-backtest-engine` installs from KaxaNuk's licensed index rather
> than PyPI (see `README.md`). The cell guards the import so the notebook stays runnable without a
> licence — it reports and skips rather than raising, and every portfolio deliverable above is
> produced either way.

In [ ]:
import importlib.util
import logging
import os

import dotenv

dotenv.load_dotenv(REPO_ROOT / "Config" / ".env")

BENCHMARK_TICKERS = backtest_engine.BENCHMARK_TICKERS
ENGINE_AVAILABLE = importlib.util.find_spec("kaxanuk.backtest_engine") is not None

if not ENGINE_AVAILABLE:
    print("kaxanuk-backtest-engine is not installed - skipping the backtest.")
    print("  Install it from the licensed index (see README.md), then re-run this cell.")
    print("\nEverything in Portfolio/ is already written and does not depend on the engine.")
    benchmark_run = None
else:
    assert os.getenv("KNBE_API_KEY_KAXANUK"), (
        "KNBE_API_KEY_KAXANUK not in the environment; add it to Config/.env"
    )

    # The engine validates that every benchmark spans the whole window, so the end is clipped to
    # the shortest series up front rather than discovered as a BenchmarkAlignmentError.
    # A clone without the hand-supplied KN600.csv measures against SPY and QQQ instead of
    # failing; the helper says which ones it dropped and why.
    benchmark_tickers = backtest_engine.available_benchmarks(MARKET_DATA_DIR, BENCHMARK_TICKERS)
    benchmark_last_dates = {
        ticker: pandas.read_csv(
            MARKET_DATA_DIR / f"{ticker}.csv", usecols=[DATE_COLUMN], parse_dates=[DATE_COLUMN],
        )[DATE_COLUMN].max()
        for ticker in benchmark_tickers
    }
    backtest_end = min(REBALANCE_DATES[-1], *benchmark_last_dates.values())

    print("Benchmark coverage ends:")
    for ticker, last_date in benchmark_last_dates.items():
        marker = "  <- binding" if last_date == backtest_end else ""
        print(f"  {ticker:<6} {last_date.date()}{marker}")
    print(f"\nBacktest window: {REBALANCE_DATES[0].date()} -> {backtest_end.date()}")

    engine_settings = backtest_engine.EngineSettings(
        market_data_directory=MARKET_DATA_DIR,
        portfolio_directory=PORTFOLIO_DIR,
        output_directory=BACKTEST_DIR,
        start_date=REBALANCE_DATES[0].date(),
        end_date=backtest_end.date(),
        benchmark_tickers=benchmark_tickers,
    )
    # `engine_weights` is the very file written in section 3.1 -- the engine reads that CSV.
    benchmark_run = backtest_engine.run_variant(
        "portfolio_weights",
        engine_weights,
        engine_settings,
    )

    print(f"\nResults: {benchmark_run.workbook_path.name}\n")
    print(benchmark_run.summary.to_string())

### 4.1 · The record, drawn from the engine's own daily series

Both charts read `portfolio_bench_total_value` out of the workbook above, so what is plotted is
exactly what the engine simulated — not a re-derivation of it.

In [ ]:
if benchmark_run is None:
    print("No engine run - nothing to chart.")
else:
    values = benchmark_run.daily_values
    equity = values / values.iloc[0]
    equity.columns = [column.replace("_Value", "") for column in equity.columns]
    # The engine reports both the invested book and the book including its cash reserve; the
    # second is the one that corresponds to the capital actually committed.
    equity = equity.drop(columns=["Portfolio"], errors="ignore")
    equity = equity.rename(columns={"Total_Portfolio": "Experiment 1"})
    drawdown = equity / equity.cummax() - 1.0

    figure, axes_grid = matplotlib.pyplot.subplots(
        2, 1, figsize=(12, 7.4), sharex=True, gridspec_kw={"height_ratios": [2, 1]},
    )
    palette = (SERIES_BLUE, INK_SECONDARY, "#9db4cc", "#2f9e6b")
    for column, color in zip(equity.columns, palette):
        axes_grid[0].plot(equity.index, equity[column], linewidth=1.6, color=color, label=column)
        axes_grid[1].plot(drawdown.index, drawdown[column], linewidth=1.1, color=color)

    axes_grid[0].set_yscale("log")
    axes_grid[0].legend(frameon=False, fontsize=9)
    style_axes(
        axes_grid[0], "Growth of 1.00, net of commission",
        subtitle="log scale; KaxaNuk Backtest Engine daily marks",
        ylabel="multiple of capital",
    )
    axes_grid[1].yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
    style_axes(axes_grid[1], "Drawdown", ylabel="from peak")
    figure.tight_layout()
    figure.savefig(CHART_DIR / "engine_equity_curve.png")
    matplotlib.pyplot.show()

    annual = equity.resample("YE").last().pct_change()
    annual.iloc[0] = equity.resample("YE").last().iloc[0] - 1.0
    annual.index = annual.index.year
    print("Calendar-year returns:")
    print(annual.map(lambda value: f"{value:.1%}" if pandas.notna(value) else "-").to_string())

---

## 5 · Attribution — KaxaNuk Attribution Analysis

The backtest says *how much* the book made; attribution says **where it came from**:

- **Brinson-Fachler** splits active return into an **allocation** effect — being overweight the
  right groups — and a **selection** effect, picking the right securities inside them.
- **A factor model** regresses the book against factor returns and leaves a residual: the part the
  factors cannot explain.

This stage exists to answer one blunt question: **is this book the signal, or a factor exposure
wearing the signal's name?** Expect the answer to be partial. An *absolute* rule — a security judged
against its own history — is close to invisible to a factor model built on *relative* factors, so a
book can beat every benchmark while the model assigns ~0% to the factor its thesis is named after.
That is a finding, not a failure; the follow-up is the counterfactual book with the signal switched
off, and the selection / sizing / timing decomposition by counterfactual books.

### The two benchmark inputs, built from the supplied index data

The library wants the benchmark as **weights** plus a **daily return series**, both in the
horizontal `Ticker × dates` layout its loader auto-detects, with no nulls. Neither exists in that
shape, so both are derived from the index files in `Data/Curator/Benchmarks/`:

| Library input | Built from | Note |
| --- | --- | --- |
| `benchmark_weights.csv` | the index holdings file | daily constituent weights; rows already sum to 1.0 |
| `benchmark_returns.csv` | the index returns file | transposed to a single row, named and date-parsed per the declaration |

**The shaping is not in this notebook.** Both builders, the file names and the date convention live
in **`Experiments/attribution_analysis.py`**, because getting the layout wrong makes the loader
mis-detect the table's orientation rather than fail — which produces a transposed attribution and no
error. Switching to a different index is an edit there; no code below names a file.

**What binds the window.** `start_date="auto"` intersects the factor files and the benchmark
holdings, so the attribution describes a *shorter* period than the backtest above. They are not
directly comparable, and that is a property of the inputs, not a bug. Record both windows in
`FINDINGS_1.md`.

Two honest caveats to carry into any reading of the numbers:

1. The library prices only the tickers in *our* portfolio, so benchmark constituents we do not hold
   arrive without returns. The allocation/selection split is indicative, not exact.
2. Group buckets come from a `*_current` column, which is **today's** classification. Every period
   before a reclassification is misattributed — see the warning in `Data/refinery.py`.

In [ ]:
ATTRIBUTION_DASHBOARD_PORT = 8051
FACTOR_MODELS_DIR = REPO_ROOT / "Data" / "Curator" / "Factors"

# What this stage has been given, and what it is missing. Asked before anything is imported, so a
# clone with no licence pays nothing to find out that it has no licence.
attribution_inputs = attribution_analysis.available_inputs(BENCHMARK_DIR, FACTOR_MODELS_DIR)
print(attribution_inputs.explain())

if not attribution_inputs.ready:
    print("  Steps 1-5 are unaffected. Everything in Portfolio/ and Backtest/ still stands.")
else:
    benchmark_weights_frame = attribution_analysis.build_benchmark_weights(
        attribution_inputs.holdings_path,
        BENCHMARK_DIR / f"{attribution_analysis.BENCHMARK_WEIGHTS_NAME}.csv",
    )
    benchmark_returns_frame = attribution_analysis.build_benchmark_returns(
        attribution_inputs.returns_path,
        BENCHMARK_DIR / f"{attribution_analysis.BENCHMARK_RETURNS_NAME}.csv",
    )
    print(f"benchmark_weights.csv : {benchmark_weights_frame.shape[0]:,} tickers x"
          f" {benchmark_weights_frame.shape[1]:,} dates")
    print(f"benchmark_returns.csv : 1 x {benchmark_returns_frame.shape[1]:,} dates")
    print(f"factor files          : {len(attribution_inputs.factor_files)}"
          "  <- record this count in FINDINGS_1.md")

    unpriced = set(benchmark_weights_frame.index) - set(engine_weights.index)
    print(f"\nBenchmark names the library cannot price: {len(unpriced):,} of"
          f" {benchmark_weights_frame.shape[0]:,}"
          f" ({len(unpriced) / benchmark_weights_frame.shape[0]:.0%})"
          " - it loads prices only for tickers in our own book,")
    print("so read Brinson-Fachler as indicative rather than exact.\n")

    from kaxanuk.attribution_analysis.entities.configuration import (
        Configuration as AttributionConfiguration,
    )
    from kaxanuk.attribution_analysis.performance_attribution import main as attribution_main

    assert os.getenv("KNAA_API_KEY_KAXANUK"), (
        "KNAA_API_KEY_KAXANUK not in the environment; add it to Config/.env"
    )

    attribution_configuration = AttributionConfiguration(
        input_directory=str(EXPERIMENT_DIR),
        weights_portfolio_directory=str(PORTFOLIO_DIR),
        weights_benchmark_directory=str(BENCHMARK_DIR),
        benchmark_returns_directory=str(BENCHMARK_DIR),
        investable_assets_directory=str(MARKET_DATA_DIR),
        factor_returns_by_factor_directory=str(FACTOR_MODELS_DIR),
        portfolio_file_name="portfolio_weights",
        benchmark_file_name=attribution_analysis.BENCHMARK_WEIGHTS_NAME,
        benchmark_return_file_name=attribution_analysis.BENCHMARK_RETURNS_NAME,
        user_column_date=DATE_COLUMN,
        user_column_price=MARK_PRICE_COLUMN,
        market_data_input_format="csv",
        portfolio_input_format="csv",
        start_date="auto",
        end_date="auto",
        brinson_fachler_method=True,
        factor_model_method=True,
    )

    # The library reports through the logging module and returns None, so a handler on the root
    # logger is what makes its numbers visible here at all.
    logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout, force=True)
    logging.getLogger("kaxanuk.attribution_analysis").setLevel(logging.INFO)

    # Its figures are a side effect too: shown, then closed, with nothing returned to save.
    # The context manager catches them on the way past and hands `pyplot.show` back afterwards.
    with attribution_analysis.figures_saved_to(ATTRIBUTION_DIR) as attribution_figures:
        attribution_main(
            attribution_configuration,
            launch_dashboard=False,
            dashboard_port=ATTRIBUTION_DASHBOARD_PORT,
        )

    logging.getLogger().handlers.clear()  # Leave the notebook's logging as we found it.
    print(f"\nAttribution figures written to {ATTRIBUTION_DIR.relative_to(REPO_ROOT)}")
    for path in attribution_figures:
        print(f"  {path.name}  ({path.stat().st_size / 1024:,.0f} KB)")

---

## 6 · Verdict

**What the portfolio stage settled, before any engine ran.** Every prediction in
[`BLUEPRINT_1.md`](BLUEPRINT_1.md) about the *shape* of the book is answered by section 3 alone, and
answering it needed no backtest. Read the invested-share chart and the group stack together: they
say whether this book rotates between groups or is a market timer wearing several tickers.

**What is still open.** Anything about volatility, return or Sharpe needs the engine. If sections 4
and 5 reported "not installed", this notebook has produced a book and no result — which is the
honest outcome and not a failure.

> **Three sentences go here once the engine has run**: does the book work; what attribution says
> about why; what the next experiment should change. Then copy the numbers into
> [`FINDINGS_1.md`](FINDINGS_1.md) — that file is the record, this notebook is the method.

---

## Handoff

| Output | Consumed by |
| --- | --- |
| `Portfolio/portfolio_weights.csv` | the backtest engine and the attribution library |
| `Portfolio/target_weights.csv` | humans, and `FINDINGS_1.md` |
| `Portfolio/portfolio_summary.csv` | `FINDINGS_1.md` |
| `Portfolio/Charts/*.png` | `FINDINGS_1.md` |
| `Backtest/`, `Attribution/` | `FINDINGS_1.md`, and the comparison baseline for every later experiment |

Later experiments read the **same** panel, over the same window, with the same costs and the same
rebalancing convention, and change only the selection or the weighting — which is what makes the
comparison against this benchmark meaningful.

In [ ]:
summary = pandas.DataFrame({
    "metric": [
        "securities in the panel",
        "trading days",
        "book starts",
        "book ends",
        "weigher",
        "holdings (mean)",
        "invested share (mean, by trading day)",
        "rebalances",
        "rebalances per year",
        "one-way turnover per year",
        "largest single weight",
        "backtest engine available",
    ],
    "value": [
        f"{signal_matrix.shape[1]}",
        f"{len(TRADING_DAYS):,}",
        f"{REBALANCE_DATES[0].date()}",
        f"{REBALANCE_DATES[-1].date()}",
        f"{WEIGHER.__name__}",
        f"{portfolio_summary['holdings'].mean():.1f}",
        f"{daily_invested.mean():.1%}",
        f"{len(REBALANCE_DATES):,}",
        f"{len(REBALANCE_DATES) / years:.0f}",
        f"{turnover.sum() / years:.0%}",
        f"{target_weights.max().max():.1%}",
        "yes" if ENGINE_AVAILABLE else "no - see the note in section 4",
    ],
})
print(summary.to_string(index=False))

---

## Open items

| # | Item | Why it matters |
| --- | --- | --- |
| 1 | **No result without the licensed engines.** Sections 4 and 5 report and skip when `kaxanuk-backtest-engine` and `kaxanuk-attribution-analysis` are absent. | The book above is real; the performance is not measured. Predictions 1 to 3 in the blueprint stay open until they run. |
| 2 | **Turnover is target-to-target, not realised.** Between rebalances the winners drift up on their own. | The realised figure is lower. The engine's own series is the one to quote. |
| 3 | **The attribution window is shorter than the backtest**, bound by factor-file coverage — and no factor file has ever been supplied here. | The two sets of numbers describe different periods and must not be compared directly. |
| 4 | **Delisting exits use one day of hindsight.** A position is sold on the last day it still has a fill price, which is only knowable the day after. | Inert on twelve live ETFs; load-bearing on any universe that retains delisted names. |
| 5 | **Cash earns the bill yield and pays commission to get there.** `BIL` is a real instrument, so going flat is not free. | A strategy that trades to cash often is partly a bet on the front end of the curve. Attribution should be asked about it. |
| 6 | **The benchmark declines the levers it could have used** — a weight cap, a minimum holding count, and inverse-volatility sizing. All three exist in `portfolio_construction.py`, switched off. | Each is a later experiment, and each has to beat this book to earn its place. |